<a href="https://colab.research.google.com/github/nakyung55/FintechAI/blob/main/2_%EC%B9%B4%EB%93%9C%EA%B1%B0%EB%9E%98%EC%9D%B4%EC%83%81%ED%83%90%EC%A7%80%EC%99%80%EC%95%99%EC%83%81%EB%B8%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **카드거래이상탐지와앙상블**

Dataset : Credit Card Fraud Detection (https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
- 데이터는 284,807건이며, 총 31개의 컬럼으로 구성됨
- 이상거래여부(Class), 거래시간(Time)과 거래금액(Amount)을 제외하고 나머지 값은 차원축소(PCA)된 값임
- Class 컬럼이 예측을 해야하는 이상거래 여부 (보편적으로 관심있는 값을 1, 아닌 경우를 0으로 설정)

# 모델의 성능 지표 : ROC AUC
- 이진분류 문제의 예측력을 측정하기 위해 사용
- 관심있는 데이터라고 판단하는 임계값을 조정해 결과를 바꿀 수 있는 측정방법을 보완

# 앙상블 모델 생성
- Stackig 개념
- Hard vote (Majority Voting) : 모델로부터 가장 많은 표를 얻은 클래스 예측
- Soft vote (Probability Voting) : 모델에서 합산 ​​확률이 가장 큰 클래스 예측

In [ ]:
import gdown
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import lightgbm as lgb
from xgboost import XGBClassifier
import pickle

google_path = 'https://drive.google.com/uc?id='
file_id = '1cA2bkyBdPvNFX8yiL-kyczqlfv8YvjLK'
output_name = 'train.csv'
gdown.download(google_path + file_id, output_name)

train_all = pd.read_csv('train.csv')

# 6:4 데이터 분리 (Stratified)
x = train_all.drop(columns=['Class'])
y = train_all['Class']

x_train_full, x_valid_full, y_train, y_valid = train_test_split(
    x, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

print("학습 데이터 크기:", x_train_full.shape)
print("검증 데이터 크기:", x_valid_full.shape)

# Train 기준 feature 선택
numeric_cols = x_train_full.select_dtypes(include=[np.number])

corr = numeric_cols.copy()
corr['Class'] = y_train
corr = corr.corr()['Class'].abs().sort_values(ascending=False)

corr_exclude = corr.drop(['Class', 'Time', 'Amount'], errors='ignore')

top10_cols = corr_exclude.index[:10].tolist()
selected_cols = ['Time', 'Amount'] + top10_cols

print("최종 선택 컬럼:", selected_cols)

x_train = x_train_full[selected_cols]
x_valid = x_valid_full[selected_cols]

# 불균형 비율 계산
pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)

Downloading...
From (original): https://drive.google.com/uc?id=1cA2bkyBdPvNFX8yiL-kyczqlfv8YvjLK
From (redirected): https://drive.google.com/uc?id=1cA2bkyBdPvNFX8yiL-kyczqlfv8YvjLK&confirm=t&uuid=ee572570-5eec-413d-acd8-14387847c71c
To: /content/train.csv
100%|██████████| 151M/151M [00:01<00:00, 89.6MB/s]


학습 데이터 크기: (170884, 30)
검증 데이터 크기: (113923, 30)
최종 선택 컬럼: ['Time', 'Amount', 'V17', 'V14', 'V12', 'V10', 'V16', 'V3', 'V7', 'V11', 'V4', 'V18']
scale_pos_weight: 578.2677966101695


# 앙상블 대상 모델 학습

In [ ]:
# Model 1: Logistic Regression
model_1 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        solver='liblinear',
        C=1.0,
        random_state=42,
        class_weight='balanced'
    ))
])

model_1.fit(x_train, y_train)
y_pred_1 = model_1.predict_proba(x_valid)[:, 1]
print("LogisticRegression ROC AUC:",
      roc_auc_score(y_valid, y_pred_1))


# Model 2: RandomForest
model_2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    class_weight='balanced'
)

model_2.fit(x_train, y_train)
y_pred_2 = model_2.predict_proba(x_valid)[:, 1]
print("RandomForest ROC AUC:",
      roc_auc_score(y_valid, y_pred_2))


# Model 3: LightGBM
model_3 = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective='binary',
    random_state=42
)

model_3.fit(x_train, y_train)
y_pred_3 = model_3.predict_proba(x_valid)[:, 1]
print("LightGBM ROC AUC:",
      roc_auc_score(y_valid, y_pred_3))


# Model 4: GradientBoosting
model_4 = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model_4.fit(x_train, y_train)
y_pred_4 = model_4.predict_proba(x_valid)[:, 1]
print("GradientBoosting ROC AUC:",
      roc_auc_score(y_valid, y_pred_4))


# Model 5: XGBoost
model_5 = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42
)

model_5.fit(x_train, y_train)
y_pred_5 = model_5.predict_proba(x_valid)[:, 1]
print("XGBoost ROC AUC:",
      roc_auc_score(y_valid, y_pred_5))

LogisticRegression ROC AUC: 0.9699651696467715
RandomForest ROC AUC: 0.9423084837177896
[LightGBM] [Info] Number of positive: 295, number of negative: 170589
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.036787 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3060
[LightGBM] [Info] Number of data points in the train set: 170884, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001726 -> initscore=-6.360037
[LightGBM] [Info] Start training from score -6.360037
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

# 앙상블 모델 생성

In [ ]:
final_model = VotingClassifier(
    estimators=[
        ('lr', model_1),
        ('rf', model_2),
        ('lgbm', model_3),
        ('xgb', model_5),
        ('gb', model_4)
    ],
    voting='soft'
)

final_model.fit(x_train, y_train)

y_pred_final = final_model.predict_proba(x_valid)[:, 1]
score = roc_auc_score(y_valid, y_pred_final)

print('Final Ensemble ROC AUC Score =', score)

with open('my_final_model.pickle', 'wb') as f:
    pickle.dump(final_model, f)

with open('my_final_model.pickle', 'rb') as f:
    loaded_model = pickle.load(f)

y_loaded = loaded_model.predict_proba(x_valid)[:, 1]
print('Loaded Model ROC AUC Score =',
      roc_auc_score(y_valid, y_loaded))

[LightGBM] [Info] Number of positive: 295, number of negative: 170589
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018345 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3060
[LightGBM] [Info] Number of data points in the train set: 170884, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001726 -> initscore=-6.360037
[LightGBM] [Info] Start training from score -6.360037
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

# 개발 방향
- train/valid 6:4 분리
- Logistic, RF, LGBM, GB, XGB 개별 학습
- Soft Voting 앙상블 구성, 성능이 좋은 모델에 높은 weight 부여


# 결과 해석
- 모델이 정상 거래와 사기 거래를 약 97% 확률로 올바르게 순위화함
- 선형 분리가 잘 되는 데이터 구조임
- 선택한 상관 상위 변수 4개와 Time, Amount 만으로도 설명력이 있음
- 극단적 불균형 데이터로 사기 비율이 매우 낮기 때문에 ROC AUC가 높아도 Recall은 낮을 수 있음 -> 실제 사기 탐지 성능은 추가 검증 필요